In [1]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.4
)
print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [3]:
itinerary_prompt = ChatPromptTemplate.from_template("""
You are VoyageAI, an intelligent multi-agent travel planner.

Use the travel knowledge context below to create a practical travel plan.

User Request:
{user_query}

Retrieved Travel Knowledge:
{context}

Create a clear travel plan with the following sections:

1. Recommended Destination
2. Why this destination matches the user
3. Suggested Trip Duration
4. Day-wise Itinerary
5. Food Recommendations
6. Stay Recommendation
7. Best Time to Visit
8. Travel Tips

Important rules:
- Use only the provided context.
- Do not invent exact hotel names unless they are present in the context.
- If exact information is missing, give general suggestions.
- Keep the answer beginner-friendly and practical.
""")

In [4]:
rag_chain = itinerary_prompt | groq_llm | StrOutputParser()

print("Groq RAG generation chain created.")

Groq RAG generation chain created.


In [5]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5543.18it/s]


Embedding model loaded successfully.


In [7]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 10


In [8]:
def get_retrieval_context(query: str, k: int = 3):
    docs = vector_store.similarity_search(query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [9]:
user_query = "Plan a trip for someone who wants beaches and seafood"

context = get_retrieval_context(
    query=user_query,
    k=3
)

final_plan = rag_chain.invoke({
    "user_query": user_query,
    "context": context
})

print(final_plan)

**Travel Plan: Beaches and Seafood**

### 1. Recommended Destination
The recommended destination for this trip is Goa, India.

### 2. Why this destination matches the user
Goa matches the user's request because it is one of India's most popular beach destinations, known for its beautiful beaches and delicious seafood. The city offers a relaxed coastal culture, making it perfect for those looking to unwind by the sea and enjoy local seafood delicacies.

### 3. Suggested Trip Duration
A typical trip duration for Goa is 3 to 5 days, which should provide enough time to explore the beaches, try the local seafood, and experience the nightlife without feeling rushed.

### 4. Day-wise Itinerary
- **Day 1:** Arrive in Goa and check into your accommodation. Spend the day relaxing at Baga Beach or Calangute Beach. In the evening, explore the local nightlife.
- **Day 2:** Visit Anjuna Beach and Fort Aguada. Enjoy water activities or simply soak up the sun. In the evening, head to Panjim to explore